# Real time data
## tfi-gtfs project
### API Key

To obtain your own API key, go to the [NTA Developer Portal](https://developer.nationaltransport.ie/) and create an account by clicking the **Sign Up** button.

You can register using your `@factored.ai` email address. Once logged in, navigate to the **Products** tab and select **GTFS Realtime**.  
Choose a name for the product and subscribe to it. After subscribing, you’ll find your two API keys under the **Profile** tab.

---

### Storing your API Key (Best Practice)

It is considered best practice not to hard-code your API key directly in your codebase.  
Instead, store it in your `local_settings.py` file and import it when needed:


```python
# local_settings.py
API_KEY = "<your-api-key>"
```



### API Routes

This server exposes the following endpoint:

| Endpoint                | Description                  |
|------------------------|------------------------------|
| `/api/v1/arrivals`     | Returns arrival information    |

Before you can query these routes, you’ll need to start the server locally.  
Run the following commands in your terminal:

```bash
# Create and activate a virtual environment
python3 -m venv my_venv
source my_venv/bin/activate

# Install dependencies
pip install -r requirements.txt

# Run the server
python3 server.py


In [43]:
import settings
import local_settings
import gtfs

In [44]:
import requests
import pandas as pd
import json
import os
import zipfile
import io
from pathlib import Path
from datetime import datetime


In [ ]:
# This is the base URL of the server that will be used to make requests to the API
BASE_URL = "http://localhost:7341"

# Assuming your API key is stored in a local_settings.py file in the same directory
api_key = local_settings.API_KEY
header = {"X-API-KEY": api_key, "Accept": "application/json"}

try:
    # TODO:
    # 1. Create function for retrieving data from a list of stop_ids
    # 2. Create function for generating dataframe from json data
    # r = requests.get(f"{BASE_URL}/api/v1/arrivals", params={"stop": stop_list}, headers=header)
    stop_id = "1508"
    r = requests.get(f"{BASE_URL}/api/v1/arrivals?stop={stop_id}", headers=header)
    r.raise_for_status()
    data = r.json()
    print("Data was retrieved successfully")
except requests.exceptions.HTTPError as e:
    print(f'HTTP error occurred: {e}')
except requests.exceptions.ConnectionError as e:
    print(f'Connection error occurred: {e}')
except requests.exceptions.RequestException as e:
    print(f'An unexpected error occurred: {e}')

Data was retrieved successfully


In [9]:
print(f"Request data for stop {stop_id}")
data

Request data for stop 1508


{'1508': {'arrivals': [{'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T10:53:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arrival': None,
    'route': 'F1',
    'scheduled_arrival': '2025-10-28T11:03:09'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F2',
    'scheduled_arrival': '2025-10-28T11:04:47'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Charlestown',
    'real_time_arrival': None,
    'route': 'F3',
    'scheduled_arrival': '2025-10-28T11:08:16'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'Tyrrelstown',
    'real_time_arrival': None,
    'route': '40D',
    'scheduled_arrival': '2025-10-28T11:14:12'},
   {'agency': 'Bus Átha Cliath – Dublin Bus',
    'headsign': 'IKEA Ballymun',
    'real_time_arr

---
# Static

## Mobility Database API Access

This section demonstrates how to authenticate and access the [Mobility Database API](https://mobilitydatabase.org/) to fetch GTFS feed information programmatically.

---

### Step 1: Create an Account

Before you can use the API, you need to create an account:

1. Visit the [Mobility Database website](https://mobilitydatabase.org/)
2. Click on **Sign Up** or **Create Account**
3. You can register using your **`@factored.ai`** email address
4. Complete the registration process and verify your email if needed

---

### Step 2: Obtain Your Refresh Token

Once your account is created and verified:

1. Log in to your Mobility Database account
2. Navigate to your **Account** page
3. Generate or copy your **Refresh Token**
4. Store this token securely (do not commit it to version control)

**Best Practice:** Store your refresh token in a `local_settings.py` file:

```python
# local_settings.py
MOBILITY_DB_REFRESH_TOKEN = "your_refresh_token_here"
```

---

### Step 3: Generate an Access Token (POST Request)

The refresh token is used to generate a short-lived **access token** that you'll use for API requests:

- **Method:** `POST`
- **Endpoint:** `https://api.mobilitydatabase.org/v1/tokens`
- **Headers:** `Content-Type: application/json`
- **Body:** `{"refresh_token": "your_refresh_token"}`

This returns an `access_token` that is valid for a limited time (1 hour).

---

### Step 4: Make API Requests (GET Request)

Use the access token to make authenticated GET requests to fetch data:

- **Method:** `GET`
- **Headers:** `Authorization: Bearer {access_token}`
- **Example Endpoint:** `https://api.mobilitydatabase.org/v1/gtfs_feeds/{feed_id}`

---

### Available Endpoints

- **List all feeds:** `GET /v1/gtfs_feeds`
- **Get specific feed:** `GET /v1/gtfs_feeds/{feed_id}`
- **Search feeds:** `GET /v1/gtfs_feeds?filter={criteria}`

For more details, refer to the [Mobility Database API Documentation](https://mobilitydatabase.org/api-docs).




---

### Example: Fetching Transport for Ireland Data

The Transport for Ireland GTFS feed has the ID **`mdb-2364`**. You can fetch its details using the authenticated API calls demonstrated below.

In [1]:
import logging
import data_utils
import local_settings

logging.basicConfig(level=logging.INFO)

In [2]:
try:
    token = data_utils.get_access_token(local_settings.REFRESH_TOKEN)
    print('\n')    
    path = data_utils.download_gtfs_feed(token, "mdb-2364")

except Exception as e:
    logging.error(f"An error occurred: {e}")

INFO:data_utils:Generating access token using refresh token: AMf-vBxyVVWau32w9UdutGwWK3PmdAzmdUSO4KYDW2CQgg_mMcaUscSVI_RvnHc6y4XCmdCE9fxFxueQbozMEXjwJwz3iiMYxGYNRBaI8dneGGm2IXsxD9BzqlTLlCe0JWn76jl74Bgfkhp15W4qT4X-GY3iukbIi0qTJAFCfMNMEEL25dUYl05zZ7H0tf7IoqVOwio4lwcdVbkuISDhXr_QkniiPvTXn3QP4XOdHri1ZTheUlFpniazJRlIWJVgdNtufRtVDaO5h6Uc9SMG8pyCdMXt2pVP56xRZJakfLHammJ1FVmy0k56pB8bBI7yDLMmHBEIgiWelZLD7Hcy_Jrt_Hdy1M6oy-FEGi-aASg4_Mr06lxHXSPSvWopRntJHNl5DltFXfcbdfFzQl2vqZZof5chBdkzsA-dz5MJbGcsof3EtR8O5LFU1yfoXtglGUYEDXGbrpU0bhzAqOsQVGL6_IXS5q6CaA
INFO:data_utils:Access token generated successfully: eyJhbGciOiJSUzI1NiIsImtpZCI6IjU0NTEzMjA5OWFkNmJmNjEzODJiNmI0Y2RlOWEyZGZlZDhjYjMwZjAiLCJ0eXAiOiJKV1QifQ.eyJuYW1lIjoiQnJld3RvbiBMb3BlcyBNb3JhaXMiLCJwaWN0dXJlIjoiaHR0cHM6Ly9saDMuZ29vZ2xldXNlcmNvbnRlbnQuY29tL2EvQUNnOG9jTGN2aFdVYksteUotaGZ4akYxN1lvSjlScUxjMWRGUDdlYTI1Q1dyTWMzSC1IU0pFWT1zOTYtYyIsImlzcyI6Imh0dHBzOi8vc2VjdXJldG9rZW4uZ29vZ2xlLmNvbS9tb2JpbGl0eS1mZWVkcy1wcm9kIiwiYXVkIjoibW9iaWxpdHktZmVlZHMtcHJv

INFO:data_utils:Feed Name: gtfs - Transport for Ireland (TFI)
INFO:data_utils:Feed ID: mdb-2364
INFO:data_utils:Downloading GTFS feed from: https://files.mobilitydatabase.org/mdb-2364/mdb-2364-202510310113/mdb-2364-202510310113.zip
INFO:data_utils:Download successful! Extracted to: data at 2025-10-31T01:13:39.557412Z


In [3]:
dataframes = {}
for file in data_utils.get_files_from_path(path, '.txt'):
    df_name = file.replace(f'{path}/', '').replace('.txt', '')
    df = data_utils.generate_df_from_txt(file)
    dataframes[df_name] = df

for name, df in dataframes.items():
    print(f"\n{name.upper()}:")
    print("-" * 30)
    print(df.head(3))

INFO:data_utils:Found 10 files with the extension .txt in the directory data



AGENCY:
------------------------------
   agency_id           agency_name                      agency_url  \
0    7778000              Citylink        https://www.citylink.ie/   
1    7778002  Nitelink, Dublin Bus       https://www.dublinbus.ie/   
2    7778006      Go-Ahead Ireland  https://www.goaheadireland.ie/   

  agency_timezone  
0   Europe/London  
1   Europe/London  
2   Europe/London  

CALENDAR_DATES:
------------------------------
   service_id      date  exception_type
0          31  20251225               2
1          31  20251226               2
2          31  20260101               2

STOP_TIMES:
------------------------------
  trip_id arrival_time departure_time       stop_id  stop_sequence  \
0  3001_1     08:00:00       08:00:00  8390PB000913              1   
1  3001_1     08:05:00       08:05:00  8390PB000923              2   
2  3001_1     08:07:00       08:07:00  8390PB000922              3   

       stop_headsign  pickup_type  drop_off_type  timepoint  
0  T

### Exploring the GTFS Data

Now that we have the data loaded, let's explore the key datasets:

## TODO

- Create constants for file paths and GTFS feed ID
- Assess if a function is needed for the real time data extraction